In [0]:
# ============================================================
# BANGLAPAY LAKEHOUSE
# Phase 0 | Unity Catalog Foundation
# Purpose: Identify the workspace and execution principal
# ============================================================

context = spark.sql("""
    SELECT
        current_user() AS execution_user,
        current_catalog() AS current_catalog,
        current_schema() AS current_schema
""")

display(context)

In [0]:
# ============================================================
# Create the main BanglaPay Lakehouse catalog.
# IF NOT EXISTS makes this operation safe to rerun.
#
# We intentionally omit MANAGED LOCATION.
# Databricks will use the configured default managed storage.
# ============================================================

spark.sql("""
    CREATE CATALOG IF NOT EXISTS banglapay
    COMMENT 'BanglaPay MFS enterprise lakehouse catalog. Contains raw, standardized, curated and operational data.'
""")

print("Catalog check completed: banglapay")

In [0]:
# ============================================================
# Create the logical layers of the BanglaPay Lakehouse.
#
# bronze : Raw source data
# silver : Cleaned and standardized data
# gold   : Business-ready data models
# ops    : Operational metadata and ingestion support
# ============================================================

schemas = [
    {
        "name": "bronze",
        "comment": "Raw source data ingested from Kafka, PostgreSQL and MongoDB."
    },
    {
        "name": "silver",
        "comment": "Validated, cleansed, standardized and deduplicated data."
    },
    {
        "name": "gold",
        "comment": "Business-ready fact tables, dimension tables and analytical models."
    },
    {
        "name": "ops",
        "comment": "Operational metadata, ingestion support and pipeline management."
    }
]

for schema in schemas:

    sql_statement = f"""
        CREATE SCHEMA IF NOT EXISTS banglapay.{schema["name"]}
        COMMENT '{schema["comment"]}'
    """

    spark.sql(sql_statement)

    print(f"Schema ready: banglapay.{schema['name']}")

In [0]:
# ============================================================
# Create a managed Unity Catalog volume for batch landing.
#
# PostgreSQL and MongoDB extracts will land here as raw files.
# Auto Loader / Lakeflow will ingest those files into Bronze.
#
# Kafka streaming does not need this landing volume.
# ============================================================

spark.sql("""
    CREATE VOLUME IF NOT EXISTS banglapay.ops.batch_landing
    COMMENT 'Managed landing volume for raw PostgreSQL and MongoDB batch extraction files.'
""")

print("Volume check completed: banglapay.ops.batch_landing")

In [0]:
# ============================================================
# Verify catalog, schemas and volume creation.
# ============================================================

print("========== CATALOGS ==========")
display(spark.sql("""
    SHOW CATALOGS
"""))

print("========== BANGLAPAY SCHEMAS ==========")
display(spark.sql("""
    SHOW SCHEMAS IN banglapay
"""))

print("========== OPERATIONAL VOLUMES ==========")
display(spark.sql("""
    SHOW VOLUMES IN banglapay.ops
"""))